# Retrieval-Augmented Generation with ChromaDB

## Learning Goals
- Understand the architecture of RAG (indexing → retrieval → generation).
- Implement a minimal RAG pipeline with **LangChain** and **ChromaDB**.
- Compare different chunking strategies and their effect on retrieval.
- Query the vector database and inject retrieved context into the LLM prompt.

In [1]:
# %load get_llm.py
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.chat_models import ChatOllama

# Load environment variables from .env
load_dotenv()

def get_llm(provider: str = "openai"):
    """
    Return a language model instance configured for either OpenAI or Ollama.

    This function centralizes the initialization of chat-based LLMs so that 
    notebooks and applications can switch seamlessly between cloud-based models 
    (OpenAI) and local models (Ollama).

    Parameters
    ----------
    provider : str, optional
        The backend provider to use. Options are:
        - "openai": returns a ChatOpenAI instance (requires OPENAI_API_KEY in .env).
        - "ollama": returns a ChatOllama instance (requires Ollama installed locally).
        Default is "openai".

    Returns
    -------
    langchain.chat_models.base.BaseChatModel
        A chat model instance that can be invoked with messages.

    Examples
    --------
    Initialize an OpenAI model (requires API key):

    >>> llm = get_llm("openai")
    >>> llm.invoke("Hello, how are you?")

    Initialize a local Ollama model (e.g., Gemma2 2B):

    >>> llm = get_llm("ollama")
    >>> llm.invoke("Summarize the benefits of reinforcement learning.")
    """
    if provider == "openai":
        return ChatOpenAI(
            model="gpt-4o-mini",  # can also be "gpt-4.1" or "gpt-4o"
            temperature=0
        )
    elif provider == "ollama":
        return ChatOllama(
            model="gemma2:2b",   # replace with any local model installed in Ollama
            temperature=0
        )
    else:
        raise ValueError("Unsupported provider. Use 'openai' or 'ollama'.")


/home/ebezerra/anaconda3/envs/sbbd2025_course/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema import HumanMessage

load_dotenv()

llm = get_llm("openai")  # or get_llm("ollama")

## Step 1 — Load sample documents
A small text file is created and used for demonstration.

In [3]:
sample_text = """
Databases are organized collections of structured information or data, typically stored electronically in a computer system. Artificial intelligence (AI) refers to systems that can perform tasks that normally require human intelligence, such as reasoning and learning. LLM-based agents combine large language models with external tools.
"""

os.makedirs("../../data", exist_ok=True)
with open("../../data/sample_doc.txt", "w") as f:
    f.write(sample_text)

loader = TextLoader("../../data/sample_doc.txt")
docs = loader.load()

print("Loaded document:")
print(docs[0].page_content)

Loaded document:

Databases are organized collections of structured information or data, typically stored electronically in a computer system. Artificial intelligence (AI) refers to systems that can perform tasks that normally require human intelligence, such as reasoning and learning. LLM-based agents combine large language models with external tools.



## Step 2 — Chunking
The document is split into overlapping chunks to fit into the model context window.
Concretely, this code splits long documents into smaller overlapping text chunks.
This is a preprocessing step commonly used in LLM agents, especially in:

- Retrieval-Augmented Generation (RAG)

- Tool-using agents with document memory

- Long-context reasoning and planning

- LLMs work best when they receive short, focused pieces of text, rather than very long documents.

`RecursiveCharacterTextSplitter` does not blindly cut text every chunk_size characters. Instead, it tries to split text hierarchically, using separators like:

- Paragraph breaks (\n\n)

- Line breaks (\n)

- Sentences

- Words

- Characters (last resort)

Only when a unit is too large does it apply character-level splitting with overlap.

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=20)
chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))
for i, c in enumerate(chunks):
    print(f"Chunk {i}: {c.page_content}")

Number of chunks: 6
Chunk 0: Databases are organized collections of structured information or data,
Chunk 1: or data, typically stored electronically in a computer system. Artificial
Chunk 2: system. Artificial intelligence (AI) refers to systems that can perform tasks
Chunk 3: can perform tasks that normally require human intelligence, such as reasoning
Chunk 4: such as reasoning and learning. LLM-based agents combine large language models
Chunk 5: language models with external tools.


## Step 3 — Embedding + Indexing in ChromaDB
Each chunk is converted into embeddings and stored in a local ChromaDB instance.

In [7]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [8]:
# embedding_model = HuggingFaceEmbeddings(
#   model_name="sentence-transformers/all-MiniLM-L6-v2"
#)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="data/chroma_store"
)

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

## Step 4 — Retrieval
Semantic search is performed over the vectorstore.

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

query = "What is artificial intelligence?"
results = retriever.get_relevant_documents(query)

print("Retrieved chunks:")
for r in results:
    print("-", r.page_content)

## Step 5 — Generation with retrieved context
The retrieved chunks are injected into a prompt template before calling the LLM.

In [ ]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Use the context provided below to answer. "
        "If the context does not provide information to answer, say 'I dont know.'.\n"
        "BEGIN of CONTEXT\n{context}\nEND OF CONTEXT"
    ),
    ("human", "{question}")
])


def rag_pipeline(question: str):
    # 1. Retrieve relevant documents
    retrieved_docs = retriever.get_relevant_documents(question)
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    # 2. Generate answer
    chain = prompt_template | llm
    answer = chain.invoke(
        {"context": context, "question": question}
    ).content

    # 3. Nicely formatted output
    print("=" * 60)
    print("QUERY:")
    print(question)
    print("-" * 60)
    print("ANSWER:")
    print(answer)
    print("=" * 60)
    print()  # blank line for readability


In [ ]:
rag_pipeline("What is a database?")
rag_pipeline("What are LLM-based agents?")
rag_pipeline("What is artificial intelligence?")
rag_pipeline("What is Quantum Physics?")

### Reflection
- RAG decouples **knowledge storage** (ChromaDB) from **reasoning** (LLM).
- Chunking is critical: too small → fragmented context; too large → exceeds token limits.
- The vector database enables **semantic search**, not keyword search.
- This architecture is the basis for practical applications like Q&A over documents.

## Exercises
1. Replace the sample document with a **PDF loader** (e.g., `PyPDFLoader`) and index a real article.
2. Change the chunk size and observe how retrieval quality changes.
3. Experiment with different embedding models (`all-MiniLM`, `multi-qa-mpnet-base-dot-v1`).
4. Persist the ChromaDB index and reload it in a new notebook.

# Misc

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Example texts (chunks from a document)
texts = [
    "The cat sits outside.",
    "It is sunny today.",
    "The dog barks loudly."
]

# Create embedding model
embedding_model = OpenAIEmbeddings()

# Generate vector representations
vectors = embedding_model.embed_documents(texts)

print(len(vectors), "embeddings generated.")
print("Dimension of each embedding:", len(vectors[0]))

# Compute cosine similarity matrix
similarity_matrix = cosine_similarity(vectors)

# Pretty print with pandas DataFrame
df = pd.DataFrame(similarity_matrix, index=texts, columns=texts)
print("\nSimilarity matrix:")
df.round(2)

In [ ]:
from langchain.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter

# Example document
text = "The cat sits outside. It is sunny today. The dog barks loudly."
chunks = CharacterTextSplitter(chunk_size=20, chunk_overlap=5, separator=" ").split_text(text)

print("Chuncks:")
print(chunks)

# Embedding model
embedding_model = OpenAIEmbeddings()

# Store chunks + embeddings in Chroma vector DB
vectorstore = Chroma.from_texts(chunks, embedding_model)

# Example query
query = "What is the weather like?"
docs = vectorstore.similarity_search(query, k=2)

print("Query:", query)
print("Result:")
for d in docs:
    print(d.page_content)

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

text = "The cat sits outside. It is sunny today. The dog barks loudly."
splitter = CharacterTextSplitter(chunk_size=20, chunk_overlap=5, separator=" ")
chunks = splitter.split_text(text)

print(chunks)
